In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import os
import requests
import time
import json
import warnings
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (mean_squared_error, r2_score,
                             accuracy_score, classification_report,
                             confusion_matrix)
from sklearn.impute import SimpleImputer
warnings.filterwarnings("ignore")

print("All imports successful")

All imports successful


In [13]:
url = "https://SDMDataAccess.sc.egov.usda.gov/Tabular/post.rest"

def query_soil(sql, label=""):
    response = requests.post(
        url,
        data={"query": sql, "format": "JSON+COLUMNNAME"},
        timeout=30
    )
    if response.status_code == 200:
        data = response.json()
        # Convert to DataFrame
        cols = data["Table"][0]
        rows = data["Table"][1:]
        df = pd.DataFrame(rows, columns=cols)
        print(f"{label} — {len(df)} rows returned")
        return df
    else:
        print(f"Error {response.status_code}:", response.text)
        return None

In [14]:
texas_features = query_soil("""
    SELECT 
        -- 1. Basic Information
        l.areasymbol, 
        l.areaname,
        m.musym, 
        m.muname,
        c.compname,
        s.saverest,
        
        -- 2. Soil Features (Category/Attribute)
        c.taxorder,        -- Soil Type (Order)
        c.drainagecl,      -- Drainage Class (e.g., Well drained)
        c.elev_r,          -- Elevation (Representative)
        c.slope_r,         -- Slope (Representative)
        
        -- 3. Soil Horizon Numerical Data (from chorizon table)
        ch.hzdept_r,       -- Top depth of horizon
        ch.hzdepb_r,       -- Bottom depth of horizon (Depth to restrictive layer)
        ch.ph1to1h2o_r,    -- pH (1:1 Water)
        ch.om_r,           -- Organic Matter (Proxy for N-P-K)
        ch.ec_r,           -- Electrical Conductivity (Salinity)
        ch.cec7_r,         -- Cation Exchange Capacity (Nutrient storage)
        ch.awc_r,          -- Available Water Capacity (Water storage)
                                
        -- 4. Crop data
        cp.cropname,       -- Crop type (e.g., Corn, Soybeans, Cotton)
        cp.yldunits,       -- Crop yield units per area
        cp.nonirryield_r,   -- The expected yield per acre without supplemental irrigation.
        cp.irryield_r      -- The expected yield per acre with irrigation.
                      

    FROM legend l
    INNER JOIN sacatalog s ON s.areasymbol = l.areasymbol
    INNER JOIN mapunit m ON m.lkey = l.lkey
    INNER JOIN component c ON c.mukey = m.mukey
    INNER JOIN chorizon ch ON ch.cokey = c.cokey
    LEFT OUTER JOIN cocropyld cp ON cp.cokey = c.cokey

    WHERE l.areasymbol LIKE 'TX%'
    AND c.majcompflag = 'Yes'
    AND ch.hzdept_r = 0 -- Wanna see the soil where the root is in
""","texas")


# ─────────────────────────────────────────
# 4. Save to CSV
# ─────────────────────────────────────────
for df, name in [
    (texas_features, "texas"),
]:
    if df is not None:
        df.to_csv(f"{name}.csv", index=False)
        print(f"Saved {name}.csv")

texas — 49146 rows returned
Saved texas.csv


In [2]:
stations_df = pd.read_fwf("../data/ghcnd-stations.txt", header=None, 
                          widths=[11, 9, 10, 7, 3, 31, 3], 
                          names=['ID', 'lat', 'lon', 'elev', 'state', 'name', 'gsn'])

print(stations_df)

                 ID      lat      lon    elev state                   name  \
0       ACW00011604  17.1167 -61.7833    10.1   NaN  ST JOHNS COOLIDGE FLD   
1       ACW00011647  17.1333 -61.7833    19.2   NaN               ST JOHNS   
2       AE000041196  25.3330  55.5170    34.0   NaN    SHARJAH INTER. AIRP   
3       AEM00041194  25.2550  55.3640    10.4   NaN             DUBAI INTL   
4       AEM00041217  24.4330  54.6510    26.8   NaN         ABU DHABI INTL   
...             ...      ...      ...     ...   ...                    ...   
129652  ZI000067969 -21.0500  29.3670   861.0   NaN         WEST NICHOLSON   
129653  ZI000067975 -20.0670  30.8670  1095.0   NaN               MASVINGO   
129654  ZI000067977 -21.0170  31.5830   430.0   NaN          BUFFALO RANGE   
129655  ZI000067983 -20.2000  32.6160  1132.0   NaN               CHIPINGE   
129656  ZI000067991 -22.2170  30.0000   457.0   NaN             BEITBRIDGE   

        gsn  
0       NaN  
1       NaN  
2        GS  
3      

In [3]:
stations_df_tx = stations_df[stations_df['state'] == 'TX']
print(stations_df_tx)

                 ID      lat       lon   elev state                   name  \
92429   US1TXAC0002  33.8281  -98.5492  305.7    TX  WICHITA FALLS 5.2 SSW   
92430   US1TXAC0003  33.5838  -98.6298  323.7    TX    ARCHER CITY 0.7 SSW   
92431   US1TXAC0005  33.7762  -98.5350  300.2    TX    WICHITA FALLS 8.5 S   
92432   US1TXAC0009  33.8328  -98.5360  300.5    TX  WICHITA FALLS 4.6 SSW   
92433   US1TXAD0002  32.0533 -102.8796  973.5    TX        ANDREWS 26.7 SW   
...             ...      ...       ...    ...   ...                    ...   
128639  USW00093955  33.6333  -95.4500  166.7    TX                  PARIS   
128645  USW00093983  31.7797  -95.7064  128.9    TX      PALESTINE MUNI AP   
128646  USW00093984  31.1500  -97.4167  207.9    TX   TEMPLE DRAUGHTON MIL   
128647  USW00093985  32.7817  -98.0603  287.1    TX       MINERAL WELLS AP   
128649  USW00093987  31.2358  -94.7547   87.2    TX  LUFKIN ANGELINA CO AP   

        gsn  
92429   NaN  
92430   NaN  
92431   NaN  
92432  

In [4]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

def fetch_fips(row):
    id, lat, lon = row['ID'], row['lat'], row['lon']
    url = f"https://geo.fcc.gov/api/census/area?lat={lat}&lon={lon}&format=json"
    response = requests.get(url).json()
    if not response['results']:
        return None
    fips_code = response['results'][0]['county_fips']
    return {'ID': id, 'lat': lat, 'lon': lon, 'fips_code': fips_code}

results = []
rows = [row for _, row in stations_df_tx.iterrows()]

with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {executor.submit(fetch_fips, row): row for row in rows}
    for future in tqdm(as_completed(futures), total=len(rows)):
        result = future.result()
        if result:
            results.append(result)

df_fips = pd.DataFrame(results)
print(df_fips.head())

100%|██████████| 6472/6472 [11:31<00:00,  9.36it/s]

            ID      lat       lon fips_code
0  US1TXAG0004  31.2058  -94.3357     48005
1  US1TXAG0001  31.3213  -94.8445     48005
2  US1TXAC0005  33.7762  -98.5350     48009
3  US1TXAG0005  31.3212  -94.7211     48005
4  US1TXAD0002  32.0533 -102.8796     48495


In [5]:
combined_df = pd.merge(stations_df_tx, df_fips, on='ID')
combined_df.head()

,ID,lat_x,lon_x,elev,state,name,gsn,lat_y,lon_y,fips_code
0,US1TXAC0002,33.8281,-98.5492,305.7,TX,WICHITA FALLS 5.2 SSW,NaN,33.8281,-98.5492,48009
1,US1TXAC0003,33.5838,-98.6298,323.7,TX,ARCHER CITY 0.7 SSW,NaN,33.5838,-98.6298,48009
2,US1TXAC0005,33.7762,-98.5350,300.2,TX,WICHITA FALLS 8.5 S,NaN,33.7762,-98.5350,48009
3,US1TXAC0009,33.8328,-98.5360,300.5,TX,WICHITA FALLS 4.6 SSW,NaN,33.8328,-98.5360,48009
4,US1TXAD0002,32.0533,-102.8796,973.5,TX,ANDREWS 26.7 SW,NaN,32.0533,-102.8796,48495


In [6]:
# Get list of Station IDs located in Texas (TX)
texas_station_ids = combined_df['ID'].unique()

# 2. Process the 2025 massive CSV with the TX filter
url = "https://www.ncei.noaa.gov/pub/data/ghcn/daily/by_year/2025.csv.gz"
cols = ['ID', 'date', 'element', 'value', 'm_flag', 'q_flag', 's_flag', 'obs_time']
target_elements = ['PRCP', 'TMAX', 'TMIN']

df_chunk = pd.read_csv(url, compression='gzip', names=cols, chunksize=200000, low_memory=False)

texas_data_list = []

print("Filtering 2025 data for Texas stations only...")

for chunk in df_chunk:
    # Filter by BOTH: 1. Element Type AND 2. Texas Station IDs
    is_target_element = chunk['element'].isin(target_elements)
    is_texas_ID = chunk['ID'].isin(texas_station_ids)
    
    target = chunk[is_target_element & is_texas_ID].copy()
    texas_data_list.append(target)


# Final Concatenation
texas_2025 = pd.concat(texas_data_list)

# Merge fips_code from combined_df by ID
texas_2025 = texas_2025.merge(combined_df[['ID', 'fips_code']], on='ID', how='left')

print(f"Finished! Final record count for Texas: {len(texas_2025)}")
print(texas_2025)

Filtering 2025 data for Texas stations only...
Finished! Final record count for Texas: 921755
                 ID      date element  value m_flag q_flag s_flag  obs_time  \
0       USW00023091  20250101    TMAX    139    NaN    NaN      W       NaN   
1       USW00023091  20250101    TMIN     -5    NaN    NaN      W       NaN   
2       USW00023091  20250101    PRCP      0    NaN    NaN      W       NaN   
3       USW00023906  20250101    TMAX    187    NaN    NaN      R       NaN   
4       USW00023906  20250101    TMIN    106    NaN    NaN      R       NaN   
...             ...       ...     ...    ...    ...    ...    ...       ...   
921750  US1TXWT0030  20251231    PRCP      0    NaN    NaN      N     700.0   
921751  US1TXWT0031  20251231    PRCP      0    NaN    NaN      N     700.0   
921752  US1TXWT0032  20251231    PRCP      0    NaN    NaN      N     900.0   
921753  US1TXYK0001  20251231    PRCP      0    NaN    NaN      N     700.0   
921754  US1TXZP0002  20251231    PRCP

In [18]:
texas_2025['date'] = pd.to_datetime(texas_2025['date'], format='%Y%m%d')
texas_2025['month'] = texas_2025['date'].dt.month

elements = ['PRCP', 'TMAX', 'TMIN']
texas_filtered = texas_2025[texas_2025['element'].isin(elements)]

texas_weather_2025 = texas_filtered.pivot_table(
    index='fips_code', 
    columns=['element', 'month'], 
    values='value'  # replace 'value' with your actual values column name
)

# Flatten multi-level columns → e.g. PRCP_1, TMAX_1, TMIN_1
texas_weather_2025.columns = [f'{elem.lower()}_{m}' for elem, m in texas_weather_2025.columns]
texas_weather_2025 = texas_weather_2025.reset_index()

element_cols = [col for col in texas_weather_2025.columns if col.startswith(('prcp_', 'tmax_', 'tmin_'))]
texas_weather_2025[element_cols] = texas_weather_2025[element_cols] / 10

# 4. Export to CSV
texas_weather_2025.to_csv("texas_weather_2025.csv")

print("--- Monthly Weather Summary 2025 with Month Column ---")
# print(texas_weather_2025)
print(texas_weather_2025.columns)

--- Monthly Weather Summary 2025 with Month Column ---
Index(['fips_code', 'prcp_1', 'prcp_2', 'prcp_3', 'prcp_4', 'prcp_5', 'prcp_6',
       'prcp_7', 'prcp_8', 'prcp_9', 'prcp_10', 'prcp_11', 'prcp_12', 'tmax_1',
       'tmax_2', 'tmax_3', 'tmax_4', 'tmax_5', 'tmax_6', 'tmax_7', 'tmax_8',
       'tmax_9', 'tmax_10', 'tmax_11', 'tmax_12', 'tmin_1', 'tmin_2', 'tmin_3',
       'tmin_4', 'tmin_5', 'tmin_6', 'tmin_7', 'tmin_8', 'tmin_9', 'tmin_10',
       'tmin_11', 'tmin_12'],
      dtype='str')


In [15]:
# Get list of Station IDs located in Texas (TX)
texas_station_ids = combined_df['ID'].unique()

cols = ['ID', 'date', 'element', 'value', 'm_flag', 'q_flag', 's_flag', 'obs_time']
target_elements = ['PRCP', 'TMAX', 'TMIN']

texas_data_list = []

for year in range(2015, 2026):
    url = f"https://www.ncei.noaa.gov/pub/data/ghcn/daily/by_year/{year}.csv.gz"
    print(f"Processing {year}...")

    df_chunk = pd.read_csv(url, compression='gzip', names=cols, chunksize=200000, low_memory=False)

    for chunk in df_chunk:
        is_target_element = chunk['element'].isin(target_elements)
        is_texas_ID = chunk['ID'].isin(texas_station_ids)

        target = chunk[is_target_element & is_texas_ID].copy()
        texas_data_list.append(target)

    print(f"Done {year}")

# Final Concatenation
texas_2015_2025 = pd.concat(texas_data_list, ignore_index=True)

# Merge fips_code from combined_df by ID
texas_2015_2025 = texas_2015_2025.merge(combined_df[['ID', 'fips_code']], on='ID', how='left')

print(f"Finished! Final record count for Texas (2015-2025): {len(texas_2015_2025)}")
print(texas_2015_2025)

Processing 2015...
  ✓ Done 2015
Processing 2016...
  ✓ Done 2016
Processing 2017...
  ✓ Done 2017
Processing 2018...
  ✓ Done 2018
Processing 2019...
  ✓ Done 2019
Processing 2020...
  ✓ Done 2020
Processing 2021...
  ✓ Done 2021
Processing 2022...
  ✓ Done 2022
Processing 2023...
  ✓ Done 2023
Processing 2024...
  ✓ Done 2024
Processing 2025...
  ✓ Done 2025
Finished! Final record count for Texas (2015-2025): 10507347
                   ID      date element  value m_flag q_flag s_flag  obs_time  \
0         USW00023091  20150101    TMAX    -38    NaN    NaN      W       NaN   
1         USW00023091  20150101    TMIN    -60    NaN    NaN      W       NaN   
2         USW00023091  20150101    PRCP      8    NaN    NaN      W       NaN   
3         USW00023906  20150101    TMAX     76    NaN    NaN      R       NaN   
4         USW00023906  20150101    TMIN     59    NaN    NaN      R       NaN   
...               ...       ...     ...    ...    ...    ...    ...       ...   
10507342 

In [17]:
texas_2015_2025['date'] = pd.to_datetime(texas_2015_2025['date'], format='%Y%m%d')
texas_2015_2025['month'] = texas_2015_2025['date'].dt.month

elements = ['PRCP', 'TMAX', 'TMIN']
texas_filtered = texas_2015_2025[texas_2015_2025['element'].isin(elements)]

texas_weather = texas_filtered.pivot_table(
    index='fips_code', 
    columns=['element', 'month'], 
    values='value',
    aggfunc='mean'
)

# Flatten multi-level columns → e.g. prcp_1, tmax_1, tmin_1
texas_weather.columns = [f'{elem.lower()}_{m}' for elem, m in texas_weather.columns]
texas_weather = texas_weather.reset_index()

# Divide all element columns by 10
element_cols = [col for col in texas_weather.columns if col.startswith(('prcp_', 'tmax_', 'tmin_'))]
texas_weather[element_cols] = texas_weather[element_cols] / 10

# Export to CSV
texas_weather.to_csv("texas_weather_2015_2025.csv", index=False)

print("--- Monthly Weather Averages 2015-2025 ---")
print(texas_weather.columns)

--- Monthly Weather Averages 2015-2025 ---
Index(['fips_code', 'prcp_1', 'prcp_2', 'prcp_3', 'prcp_4', 'prcp_5', 'prcp_6',
       'prcp_7', 'prcp_8', 'prcp_9', 'prcp_10', 'prcp_11', 'prcp_12', 'tmax_1',
       'tmax_2', 'tmax_3', 'tmax_4', 'tmax_5', 'tmax_6', 'tmax_7', 'tmax_8',
       'tmax_9', 'tmax_10', 'tmax_11', 'tmax_12', 'tmin_1', 'tmin_2', 'tmin_3',
       'tmin_4', 'tmin_5', 'tmin_6', 'tmin_7', 'tmin_8', 'tmin_9', 'tmin_10',
       'tmin_11', 'tmin_12'],
      dtype='str')
